# LG-04: 人机协作与 Hooks

> **阶段**: LG-04 | **难度**: 中级 | **预计时长**: 4-5 小时（含 2 小时实战） | **依赖**: LG-01, LG-02, LG-03

## 学习目标
- 掌握 `interrupt()` 原语实现人工审批/干预
- 理解 `interrupt_before` / `interrupt_after` 的配置方式
- 掌握 `Command(resume=...)` 恢复中断的执行
- 掌握 Pre/Post Model Hooks 实现消息预处理和后处理
- 理解 `llm_input_messages` 临时注入上下文的机制
- 能够设计完整的审批工作流

---

In [ ]:
# 安装依赖（如未安装请取消注释）
# !pip install -U langgraph langchain langchain-openai

## 1. 为什么需要人机协作？

AI 不是万能的。在以下场景需要人工介入：
- **高风险决策**（资金转账、内容发布）
- **敏感内容审核**（医疗、法律建议）
- **质量把关**（营销文案、合同生成）
- **异常处理**（AI 无法判断的情况）

**生活化类比**：自动扶梯是自动运行的，但每部扶梯都有一个红色的急停按钮。紧急情况下，任何人都可以按下它，让扶梯停下来。

HITL（Human-in-the-Loop）就是这个急停按钮——图在正常执行，但在关键节点，它可以停下来等人类输入，然后再决定是继续、修改还是终止。

LangGraph 提供 `interrupt()` 原语，让图可以在任意节点暂停等待人工输入。

## 2. interrupt() 原语

`interrupt()` 会暂停图执行，等待外部通过 `Command(resume=...)` 恢复。

**类比**：`interrupt()` 就像你在做菜时突然发现酱油没了——你停下来，喊一嗓子『谁帮我买瓶酱油』，然后站在原地等。菜还在锅里（State 已保存），但你不再执行下一步。

In [ ]:
from typing import TypedDictfrom langgraph.graph import StateGraph, START, ENDfrom langgraph.types import interrupt, Commandfrom langgraph.checkpoint.memory import MemorySaverfrom langchain_core.messages import HumanMessage, AIMessage# ==========================================# 定义 State# ==========================================class ContentState(TypedDict):    draft: str    risk_score: float    approved: bool    feedback: str    final_content: strdef generate_draft(state: ContentState) -> dict:    """AI 生成营销文案草稿"""    print("[generate_draft] AI 生成营销文案草稿...")    draft = "限时特惠！全场商品五折起,错过再等一年！"    return {"draft": draft}def auto_review(state: ContentState) -> dict:    """自动审核：检测风险关键词"""    draft = state["draft"]    print(f"[auto_review] 自动审核: {draft}")    # 简单规则：包含"五折"等夸张用语 → 高风险    risk = 0.8 if "五折" in draft else 0.2    return {"risk_score": risk}def human_approval(state: ContentState) -> dict:    """人工审批节点：高风险内容需要人工确认"""    risk = state["risk_score"]    print(f"[human_approval] 风险评分: {risk}")        if risk > 0.5:        # ==========================================        # 触发中断！图在这里暂停,等待人工输入        # ==========================================        feedback = interrupt({            "action": "requires_approval",            "draft": state["draft"],            "risk_score": risk,            "message": "该内容被标记为高风险,请审核员确认"        })        print(f"[human_approval] 收到审核结果: {feedback}")        return {            "approved": feedback.get("approved", False),            "feedback": feedback.get("comment", "")        }    else:        # 低风险,自动通过        return {"approved": True, "feedback": "自动通过"}def publish_content(state: ContentState) -> dict:    """发布内容"""    if state["approved"]:        print("[publish_content] 内容已发布！")        return {"final_content": state["draft"]}    else:        print("[publish_content] 内容被拒绝发布")        return {"final_content": f"[已拒绝] {state['feedback']}"}# ==========================================# 构建图# ==========================================builder = StateGraph(ContentState)builder.add_node("generate_draft", generate_draft)builder.add_node("auto_review", auto_review)builder.add_node("human_approval", human_approval)builder.add_node("publish_content", publish_content)builder.add_edge(START, "generate_draft")builder.add_edge("generate_draft", "auto_review")builder.add_edge("auto_review", "human_approval")builder.add_edge("human_approval", "publish_content")builder.add_edge("publish_content", END)# 必须配置 checkpointer,否则中断后 State 会丢失！memory = MemorySaver()graph = builder.compile(checkpointer=memory)print("ContentGuard 图编译成功！")print("流程: 生成草稿 → 自动审核 → [interrupt]人工审批 → 发布")

In [ ]:
from IPython.display import Image, display

# 可视化图结构
png_bytes = graph.get_graph().draw_mermaid_png()
display(Image(data=png_bytes))

## 3. 模拟人机协作流程

由于环境限制，我们用 print 模拟 interrupt 的行为。实际使用时：
1. 图执行到 interrupt 处暂停
2. 前端显示审批界面
3. 审核员点击通过/拒绝
4. 调用 `graph.invoke(Command(resume={...}), config)` 恢复

In [ ]:
print("=" * 60)print("模拟 ContentGuard 人机协作流程")print("=" * 60)print()print("步骤 1: AI 生成草稿")print("  draft = '限时特惠！全场商品五折起,错过再等一年！'")print()print("步骤 2: 自动审核")print("  risk_score = 0.8  (高风险)")print()print("步骤 3: 触发人工审批 (interrupt)")print("  - 前端弹出审批对话框")print("  - 审核员查看草稿和风险评分")print("  - 审核员选择：通过 / 拒绝 / 编辑")print()print("步骤 4: 恢复执行 (Command.resume)")print("  graph.invoke(")print("      Command(resume={'approved': True, 'comment': '同意发布'}),")print("      config={'configurable': {'thread_id': 'content_001'}}")print("  )")print()print("步骤 5: 发布或拒绝")print("=" * 60)print()print("关键理解:")print("  - 中断后,之前节点的执行结果还在（已保存到 Checkpoint）")print("  - Command(resume) 的值被中断的节点接收,作为 interrupt() 的返回值")print("  - 必须有 checkpointer,否则中断后 State 丢失,无法恢复！")

## 4. interrupt_before / interrupt_after 配置

可以在编译时全局配置中断点，**无需在节点函数中写 interrupt()**。

| 配置方式 | 作用时机 | 使用场景 |
|---------|---------|---------|
| `interrupt_before` | 节点执行前 | 审批员先看输入再决定让不让 AI 处理 |
| `interrupt_after` | 节点执行后 | AI 生成完后让人类确认 |

In [ ]:
print("编译时配置中断点:")print()print("# 方式 1：在节点执行前中断")print("graph = builder.compile(")print("    checkpointer=memory,")print("    interrupt_before=['publish']  # 发布前必须审批")print(")")print()print("# 方式 2：在节点执行后中断")print("graph = builder.compile(")print("    checkpointer=memory,")print("    interrupt_after=['generate']  # 生成后让人确认")print(")")print()print("# 方式 3：组合使用")print("graph = builder.compile(")print("    checkpointer=memory,")print("    interrupt_before=['publish'],     # 发布前审批")print("    interrupt_after=['generate']      # 生成后确认")print(")")print()print("=" * 60)print("interrupt_before vs interrupt_after 对比:")print("=" * 60)print()comparison = [    ("执行时机", "节点函数执行前", "节点函数执行后"),    ("State 状态", "节点输入状态", "节点输出状态（已更新）"),    ("典型场景", "审批员先看输入", "AI生成后让人确认"),    ("代码侵入", "无需改节点代码", "无需改节点代码"),    ("灵活性", "声明式,全局配置", "声明式,全局配置"),]for item, before, after in comparison:    print(f"{item:<12} | {before:<25} | {after}")

## 5. Command(resume=...) 恢复执行

人类输入后，用 `Command(resume=value)` 继续执行。

**resume 值的传递**：被中断的节点收到 resume 值作为 `interrupt()` 的返回值。

In [ ]:
print("Command(resume) 的三种使用模式:")print("=" * 60)print()print("# 模式 1: 用户审批通过")print("graph.invoke(")print("    Command(resume={"approved": True, "comment": "同意发布"}),")print("    config={'configurable': {'thread_id': 'content_001'}}")print("")print()print("# 模式 2: 用户要求修改")print("graph.invoke(")print("    Command(resume={")print("        "decision": "revise",")print("        "feedback": "语气太正式了,改成更活泼的风格"")print("    }),")print("    config={'configurable': {'thread_id': 'content_001'}}")print("")print()print("# 模式 3: 用户拒绝")print("graph.invoke(")print("    Command(resume={"approved": False, "reason": "涉及敏感词汇"}),")print("    config={'configurable': {'thread_id': 'content_001'}}")print("")print()print("=" * 60)print("关键注意点:")print("  1. resume 可以是任意类型（dict, str, bool, list...）")print("  2. resume 值必须与被中断节点的期望输入匹配")print("  3. 必须使用同一个 thread_id 才能正确恢复")print("  4. 没有 checkpointer → 中断后 State 丢失 → 无法恢复！")

## 6. Pre/Post Model Hooks

Pre/Post Model Hooks 是 LangGraph v2 版本引入的高级特性，允许在 LLM 调用前后插入自定义处理逻辑。

**执行流程**：
```
用户输入 
  ↓
pre_model_hook (消息预处理)
  ↓
Agent Node (LLM 调用)
  ↓
post_model_hook (响应后处理)
  ↓
工具调用 / 最终输出
```

**类比**：Pre-hook 就像给 LLM 戴耳机——在它『听』问题之前，你给加点提示；Post-hook 就像给 LLM 装个过滤器——它『说』完之后，你先检查一下再放出去。

### 6.1 两种 Hook 对比

In [ ]:
print("Pre-Model Hook vs Post-Model Hook 对比:")
print("=" * 70)

comparison = [
    ("执行时机", "LLM 调用前", "LLM 调用后"),
    ("主要用途", "消息预处理", "响应后处理"),
    ("典型场景", "裁剪、摘要、上下文注入", "审批、验证、日志"),
    ("状态更新", "messages / llm_input_messages", "任意状态键"),
    ("支持中断", "❌", "✅ (HITL)"),
    ("访问 LLM 响应", "❌", "✅"),
    ("版本要求", "v2", "v2"),
]

print(f"{'特性':<18} {'Pre-Model Hook':<22} {'Post-Model Hook'}")
print("-" * 65)
for feat, pre, post in comparison:
    print(f"{feat:<18} {pre:<22} {post}")

### 6.2 Pre-Model Hook（前置钩子）

在 **LLM 调用之前** 对消息进行预处理。

In [ ]:
from typing import Dict, Anyfrom langchain_core.messages import BaseMessage, SystemMessage, RemoveMessage# ==========================================# Pre-Model Hook 函数签名# ==========================================def pre_model_hook(state: Dict[str, Any]) -> Dict[str, Any]:    """    参数:        state: 当前 graph 状态        返回:        必须包含以下至少一个键:        - messages: 更新状态中的消息列表 (永久修改)        - llm_input_messages: 仅供 LLM 使用,不更新状态 (临时上下文)    """    return {        "messages": [...],           # 可选: 更新状态        "llm_input_messages": [...], # 可选: 仅供 LLM 输入    }# ==========================================# 实战案例 1: 消息裁剪 Hook# ==========================================def trim_messages_hook(state: Dict[str, Any]) -> Dict[str, Any]:    """保留最近 10 条消息,删除其他历史"""    messages = state.get("messages", [])        if len(messages) <= 10:        return {}  # 无需处理        recent_messages = messages[-10:]        return {        # 必须先清空再添加新消息        "messages": [            RemoveMessage(id="__remove_all__"),            *recent_messages        ]    }# ==========================================# 实战案例 2: 用户偏好注入 Hook# ==========================================def inject_user_context_hook(state: Dict[str, Any]) -> Dict[str, Any]:    """临时注入用户偏好,不修改状态"""    messages = list(state.get("messages", []))    user_id = state.get("user_id")        # 模拟从 Store 读取用户偏好    user_prefs = {"language": "中文", "style": "正式"} if user_id else None        if user_prefs:        context_message = SystemMessage(            content=f"用户偏好: {user_prefs['language']}, {user_prefs['style']}"        )        return {            # 仅供本次 LLM 调用使用            "llm_input_messages": [context_message, *messages]            # 状态中的 messages 保持不变        }        return {"llm_input_messages": messages}print("Pre-Model Hook 示例定义完成")print()print("模式 1 - messages: 永久修改状态")print("模式 2 - llm_input_messages: 临时注入,不修改状态")

### 6.3 Post-Model Hook（后置钩子）

在 **LLM 调用之后、工具调用之前** 进行后处理。

In [ ]:
from langgraph.types import interrupt# ==========================================# 实战案例 1: 工具调用人工审批 (HITL)# ==========================================def approval_post_hook(state: Dict[str, Any]) -> Dict[str, Any]:    """工具调用前需要人工批准"""    messages = state.get("messages", [])    last_message = messages[-1] if messages else None        # 检查是否有工具调用    if hasattr(last_message, "tool_calls") and last_message.tool_calls:        tool_calls = last_message.tool_calls                # 触发中断,等待人工决策        decision = interrupt({            "action": "approve_tools",            "tool_calls": [                {"name": tc["name"], "args": tc["args"]}                for tc in tool_calls            ],            "message": "请确认是否执行这些工具"        })                # 根据决策修改状态        if decision.get("approved"):            return {}  # 允许继续        else:            # 拒绝工具调用            return {                "messages": [                    AIMessage(content="工具调用被拒绝")                ]            }        return {}# ==========================================# 实战案例 2: 内容安全审查# ==========================================def content_safety_hook(state: Dict[str, Any]) -> Dict[str, Any]:    """检测 LLM 输出是否包含敏感内容"""    messages = state.get("messages", [])    last_message = messages[-1] if messages else None        if not last_message:        return {}        content = getattr(last_message, "content", "")        # 敏感词列表    sensitive_words = ["敏感词", "密码", "密钥", "token"]    found = [w for w in sensitive_words if w in content]        if found:        # 替换为安全消息        return {            "messages": [                RemoveMessage(id=last_message.id),                AIMessage(content=f"抱歉,无法处理该请求。内容包含敏感信息: {found}")            ]        }        return {}# ==========================================# 实战案例 3: 统一工具审批（带危险工具检测）# ==========================================def unified_tool_approval_hook(state: Dict[str, Any]) -> Dict[str, Any]:    """    统一的工具调用审批逻辑    根据 auto 决定是否需要人工确认（auto=False 才需要）    """    messages = state.get("messages", [])    last_message = messages[-1] if messages else None    auto = state.get("auto")    tool_mode = "copilot" if (auto is None or auto) else "interactive"        # 检查工具调用    if not (hasattr(last_message, "tool_calls") and last_message.tool_calls):        return {}        tool_calls = last_message.tool_calls        # Interactive 模式需要审批    if tool_mode == "interactive":        # 危险工具列表        dangerous_tools = ["delete_file", "execute_code", "coding_tool"]                needs_approval = any(            tc["name"] in dangerous_tools             for tc in tool_calls        )                if needs_approval:            decision = interrupt({                "action": "approve_tools",                "tool_calls": tool_calls,                "config": {                    "allow_accept": True,                    "allow_reject": True,                    "allow_edit": True                }            })                        if decision.get("type") == "reject":                return {                    "messages": [                        AIMessage(content="工具调用已被用户拒绝")                    ]                }            elif decision.get("type") == "edit":                # 修改工具参数                edited_args = decision.get("args", {})                print(f"[edit] 用户修改了参数: {edited_args}")        return {}print("Post-Model Hook 示例定义完成")print()print("案例 1: approval_post_hook - 工具调用前人工审批")print("案例 2: content_safety_hook - 内容安全审查")print("案例 3: unified_tool_approval_hook - 统一工具审批 + 危险工具检测")

### 6.4 组合使用 Pre + Post Hooks

In [ ]:
# ==========================================# 组合使用 Pre + Post Hooks 的完整示例# ==========================================def combined_pre_hook(state: Dict[str, Any]) -> Dict[str, Any]:    """统一前置处理: 消息裁剪 + 用户偏好注入"""    messages = list(state.get("messages", []))    user_id = state.get("user_id")        # 1. 消息裁剪（保留最近 15 条）    if len(messages) > 20:        # 模拟摘要        summary = f"前 {len(messages) - 15} 条消息已摘要"        messages = [            SystemMessage(content=f"历史摘要: {summary}"),            *messages[-15:]        ]        # 2. 用户偏好注入（临时上下文）    context_parts = []    if user_id:        context_parts.append(f"用户ID: {user_id}")    context_parts.append("请遵循系统化、结构化的输出风格")        context_message = SystemMessage(content="\n".join(context_parts))        return {        "llm_input_messages": [context_message, *messages]    }def combined_post_hook(state: Dict[str, Any]) -> Dict[str, Any]:    """统一后置处理: 内容安全 + 工具审批"""    messages = state.get("messages", [])    last_message = messages[-1] if messages else None        if not last_message:        return {}        # 1. 内容安全检查    content = getattr(last_message, "content", "")    sensitive_words = ["敏感词", "密码", "密钥"]    found = [w for w in sensitive_words if w in content]        if found:        return {            "messages": [                AIMessage(content=f"抱歉,无法处理该请求。原因: 包含敏感词 {found}")            ]        }        # 2. 工具调用审批（简化版,不实际触发 interrupt）    if hasattr(last_message, "tool_calls") and last_message.tool_calls:        tool_names = [tc["name"] for tc in last_message.tool_calls]        print(f"[post_hook] 检测到工具调用: {tool_names}")        return {}print("组合 Hooks 定义完成")print()print("Pre-Hook 职责:")print("  - 消息裁剪（防止上下文爆炸）")print("  - 用户偏好注入（临时上下文）")print()print("Post-Hook 职责:")print("  - 内容安全审查")print("  - 工具调用审批")print("  - 日志记录")

### 6.5 Hooks 最佳实践

In [ ]:
print("Hooks 最佳实践:")print("=" * 60)print()print("1. 关注点分离")print("   ✅ 好: 每个 hook 专注一件事")print("      pre_hook = trim_messages_hook")print("      post_hook = safety_check_hook")print()print("   ❌ 差: 一个 hook 做太多事")print("      def mega_hook(state):")print("          # 裁剪 + 摘要 + 审查 + 日志 + ...")print()print("2. 性能优化")print("   ✅ 好: 仅在必要时处理")print("      if len(messages) < 10: return {}")print()print("   ❌ 差: 每次都执行复杂逻辑")print("      await expensive_operation()  # 每次都调用")print()print("3. 错误处理")print("   ✅ 好: Hook 失败不中断流程")print("      try:")print("          return await process(state)")print("      except Exception as e:")print("          logger.error(f'Hook failed: {e}')")print("          return {}  # 返回空更新")print()print("4. 覆盖消息的正确方式")print("   ❌ 错误: 直接覆盖会导致状态不一致")print("      return {'messages': new_messages}")print()print("   ✅ 正确: 必须先清空再添加")print("      return {")print("          'messages': [")print("              RemoveMessage(id="__remove_all__"),")print("              *new_messages")print("          ]")print("      }")print()print("5. 至少返回一个消息键")print("   ❌ 错误: 没有返回 messages 或 llm_input_messages")print("      return {'user_id': '123'}")print()print("   ✅ 正确")print("      return {'llm_input_messages': messages, 'user_id': '123'}")

## 7. llm_input_messages：临时注入上下文

`llm_input_messages` 是 Pre-Model Hook 的一个特殊返回键，用于**临时向 LLM 注入上下文，而不修改 State**。

**核心区别**：
- `messages`：永久修改 State 中的消息列表，后续节点可见
- `llm_input_messages`：仅用于本次 LLM 调用，**不保存到 State**，后续节点不可见

In [ ]:
from typing import TypedDict, Listfrom langgraph.graph import StateGraph, START, ENDfrom langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage# ==========================================# llm_input_messages: 构建临时 LLM 输入消息（不污染 state）# ==========================================class RAGState(TypedDict):    query: str    retrieved_docs: List[str]    response: str    messages: List[BaseMessage]def retrieve_docs(state: RAGState) -> dict:    """模拟文档检索节点"""    query = state["query"]    docs = [        f"《LangGraph 实战指南》: {query} 是 LangChain 生态中的工作流编排框架...",        f"《AI Agent 设计模式》: {query} 支持持久化、人机协作和并行执行...",    ]    print(f"[retrieve_docs] 查询 '{query}' 检索到 {len(docs)} 篇文档")    return {"retrieved_docs": docs}def build_llm_input_messages(state: RAGState) -> List[BaseMessage]:    """    llm_input_messages 核心逻辑：    1. 基于 state 中的原始消息临时构建输入给 LLM 的消息列表    2. 动态注入检索结果作为临时上下文（不保存到 state）    3. 对历史消息进行过滤/截断,控制上下文长度    4. 预处理消息格式（如确保最后一条是用户消息）    """    messages = list(state.get("messages", []))        # 1. 临时注入检索结果（SystemMessage 形式,仅用于本次 LLM 调用）    docs = state.get("retrieved_docs", [])    if docs:        context = "\n".join([f"[{i+1}] {d}" for i, d in enumerate(docs)])        context_msg = SystemMessage(            content=f"以下是本次查询相关的参考资料（仅作临时参考,不记录到对话历史）：\n{context}"        )        # 将参考资料插入到用户消息之前,作为临时上下文        messages = [context_msg] + messages        # 2. 消息过滤：只保留最近 5 条消息,防止上下文过长导致 Token 溢出    original_count = len(messages)    messages = messages[-5:]    if original_count > 5:        print(f"[llm_input_messages] 消息截断: {original_count} → 5 条")        # 3. 消息预处理：如果最后一条不是用户消息,追加提示确保对话连续性    if messages and messages[-1].type != "human":        messages.append(HumanMessage(content="请基于以上上下文继续回答"))        # 4. 格式校验：过滤空消息    messages = [m for m in messages if hasattr(m, "content") and str(m.content).strip()]        return messagesdef rag_generate_node(state: RAGState) -> dict:    """RAG 生成节点：使用 llm_input_messages 进行带临时上下文的生成"""        # 构建临时 LLM 输入消息（这些消息不会保存到 state）    llm_input = build_llm_input_messages(state)        print(f"\n[rag_generate_node] State 原始消息数: {len(state.get('messages', []))}")    print(f"[rag_generate_node] LLM 实际输入消息数: {len(llm_input)}")    print("-" * 50)    for i, msg in enumerate(llm_input):        tag = "【临时上下文】" if msg.type == "system" and "参考资料" in str(msg.content) else "【原始消息】"        preview = str(msg.content)[:70].replace("\n", " ")        print(f"  [{i}] {tag} {msg.type}: {preview}...")    print("-" * 50)        # 模拟 LLM 调用    doc_titles = [d[:15] for d in state.get("retrieved_docs", [])]    response_text = f"基于《{doc_titles[0]}》等 {len(doc_titles)} 篇资料,关于「{state['query']}」的解答如下：LangGraph 是...（模拟回答）"        # 关键：只将最终 AI 响应加入 state.messages,临时 SystemMessage 不会污染 state    new_ai_message = AIMessage(content=response_text)    updated_messages = state.get("messages", []) + [new_ai_message]        return {        "response": response_text,        "messages": updated_messages    }# 构建图rag_builder = StateGraph(RAGState)rag_builder.add_node("retrieve", retrieve_docs)rag_builder.add_node("rag_generate", rag_generate_node)rag_builder.add_edge(START, "retrieve")rag_builder.add_edge("retrieve", "rag_generate")rag_builder.add_edge("rag_generate", END)rag_graph = rag_builder.compile()print("=" * 60)print("llm_input_messages 演示：临时上下文注入与消息预处理")print("=" * 60)result = rag_graph.invoke({    "query": "LangGraph 的持久化机制",    "retrieved_docs": [],    "response": "",    "messages": [        HumanMessage(content="你好"),        AIMessage(content="你好！有什么可以帮您？"),        HumanMessage(content="介绍一下 LangGraph 的持久化机制"),    ]})print(f"\n最终响应: {result['response']}")print(f"State 中最终保存的消息数: {len(result['messages'])}")print("✅ 注意：临时注入的参考资料 SystemMessage 没有被保存到 state 中")

## 8. 完整案例：ContentGuard 审批工作流

综合运用 interrupt、Hooks、llm_input_messages 实现一个完整的内容审批系统。

In [ ]:
from typing import TypedDict, Listfrom langgraph.graph import StateGraph, START, ENDfrom langgraph.types import interrupt, Commandfrom langgraph.checkpoint.memory import MemorySaverfrom langchain_core.messages import HumanMessage, AIMessage, SystemMessage, RemoveMessage# ==========================================# ContentGuard 完整审批工作流# ==========================================class ContentGuardState(TypedDict):    topic: str                    # 内容主题    draft: str                    # AI 生成的草稿    revised_draft: str            # 修改后的草稿    risk_score: float             # 风险评分    approved: bool                # 是否通过审批    feedback: str                 # 审批反馈    final_content: str            # 最终发布内容    audit_log: List[str]          # 审计日志    messages: List                # 对话历史def pre_generate_hook(state: ContentGuardState) -> Dict[str, Any]:    """    Pre-model Hook: 在生成内容前注入品牌规范    使用 llm_input_messages 临时注入,不污染 state    """    messages = state.get("messages", [])    topic = state.get("topic", "")        # 临时注入品牌规范（不保存到 state）    brand_guidelines = SystemMessage(        content="""【品牌规范 - 仅本次生成参考】- 我们是高端家电品牌,语气要优雅- 禁止使用网络梗和夸张用语- 促销信息必须准确,不得误导消费者- 当前主题: {topic}        """.format(topic=topic)    )        return {        "llm_input_messages": [brand_guidelines] + list(messages)    }def generate_content(state: ContentGuardState) -> dict:    """AI 生成营销文案"""    topic = state["topic"]    print(f"[generate] 正在为 '{topic}' 生成文案...")        # 模拟 AI 生成（实际应调用 LLM）    draft = f"【{topic}】品质生活,从选择开始。限时优惠,尊享专属折扣。"        return {        "draft": draft,        "audit_log": [f"生成草稿: {draft[:30]}..."]    }def auto_review(state: ContentGuardState) -> dict:    """自动审核：检测风险关键词"""    draft = state.get("draft", "") or state.get("revised_draft", "")        # 风险规则    high_risk_words = ["免费", "免费送", "全场一折"]    medium_risk_words = ["限时", "抢购", "秒杀"]        risk = 0.0    for word in high_risk_words:        if word in draft:            risk = 0.9            break    if risk == 0.0:        for word in medium_risk_words:            if word in draft:                risk = 0.5                break        print(f"[auto_review] 风险评分: {risk}")    return {        "risk_score": risk,        "audit_log": state.get("audit_log", []) + [f"自动审核: 风险={risk}"]    }def post_generate_hook(state: ContentGuardState) -> Dict[str, Any]:    """    Post-model Hook: 生成后进行敏感词检查    如果检测到敏感内容,直接替换    """    draft = state.get("draft", "")        # 敏感词检查    sensitive = ["敏感词", "违禁词"]    found = [w for w in sensitive if w in draft]        if found:        safe_draft = draft        for w in found:            safe_draft = safe_draft.replace(w, "***")        print(f"[post_hook] 检测到敏感词 {found},已替换")        return {"draft": safe_draft}        return {}def human_approval(state: ContentGuardState) -> dict:    """人工审批节点"""    risk = state["risk_score"]    draft = state.get("revised_draft", "") or state["draft"]        if risk > 0.3:        # 高风险或中风险,需要人工审批        feedback = interrupt({            "action": "content_approval",            "draft": draft,            "risk_score": risk,            "options": ["approve", "reject", "revise"],            "message": f"该内容风险评分为 {risk},请审核员确认"        })                decision = feedback.get("decision", "reject")                if decision == "approve":            return {                "approved": True,                "feedback": feedback.get("comment", "")            }        elif decision == "reject":            return {                "approved": False,                "feedback": feedback.get("reason", "内容被拒绝")            }        elif decision == "revise":            # 返回修改要求,让 revise 节点处理            return {                "approved": False,                "feedback": feedback.get("feedback", "需要修改"),                "audit_log": state.get("audit_log", []) + ["要求修改"]            }    else:        # 低风险,自动通过        return {"approved": True, "feedback": "自动通过"}def revise_content(state: ContentGuardState) -> dict:    """根据反馈修改内容"""    feedback = state["feedback"]    original = state["draft"]        print(f"[revise] 根据反馈修改: {feedback}")        # 模拟修改（实际应调用 LLM 重新生成）    revised = f"[已修改] {original} （根据反馈：{feedback}）"        return {        "revised_draft": revised,        "audit_log": state.get("audit_log", []) + [f"修改: {feedback}"]    }def route_by_approval(state: ContentGuardState) -> str:    """根据审批结果路由"""    if state.get("revised_draft") and not state["approved"]:        # 有修改稿但未批准 → 需要重新审批        return "re_review"    return "publish"def publish_content(state: ContentGuardState) -> dict:    """发布内容"""    if state["approved"]:        content = state.get("revised_draft", "") or state["draft"]        print(f"[publish] 内容已发布: {content[:50]}...")        return {            "final_content": content,            "audit_log": state.get("audit_log", []) + ["已发布"]        }    else:        return {            "final_content": f"[已拒绝] {state['feedback']}",            "audit_log": state.get("audit_log", []) + ["已拒绝"]        }# ==========================================# 构建完整工作流# ==========================================cg_builder = StateGraph(ContentGuardState)cg_builder.add_node("generate", generate_content)cg_builder.add_node("auto_review", auto_review)cg_builder.add_node("human_approval", human_approval)cg_builder.add_node("revise", revise_content)cg_builder.add_node("publish", publish_content)cg_builder.add_edge(START, "generate")cg_builder.add_edge("generate", "auto_review")cg_builder.add_edge("auto_review", "human_approval")# 审批后路由：通过 → 发布；拒绝但有修改 → 修改后重新审核cg_builder.add_conditional_edges("human_approval", route_by_approval, {    "re_review": "revise",    "publish": "publish"})cg_builder.add_edge("revise", "auto_review")  # 修改后重新审核cg_builder.add_edge("publish", END)cg_memory = MemorySaver()content_guard = cg_builder.compile(checkpointer=cg_memory)print("ContentGuard 完整工作流编译成功！")print()print("流程图:")print("  START → generate → auto_review → human_approval → [分支]")print("                              ↓")print("                    ┌─────────┴─────────┐")print("                    ↓                   ↓")print("                 revise(修改)      publish(发布)")print("                    ↓                   ↓")print("              auto_review(重审)       END")print("                    ↑")print("              human_approval(再审)")

In [ ]:
# 可视化完整工作流
png_bytes = content_guard.get_graph().draw_mermaid_png()
display(Image(data=png_bytes))

### ContentGuard 案例讲解话术

**Step 1 - 生成文案**：
> "首先 AI 生成营销文案。大家看 `generate` 节点——它调用 LLM 写文案。但因为我们在 compile 时配置了 `interrupt_after=["generate"]`，所以这个节点执行完后，图会自动暂停。"

**Step 2 - 人工审批**：
> "现在前端弹出了审批面板。审批员看到 AI 生成的文案：『五一大促，全场免费送』。审批员觉得『免费送』有风险，点击『修改』，输入反馈：『把免费送改成满 100 减 50』。"

**Step 3 - 恢复执行**：
> "前端发送 `Command(resume={"decision": "revise", "feedback": "..."})`。图从中断点恢复，这个 resume 值被传给下一个节点。`revise` 节点收到反馈后，调用 LLM 重新生成文案。"

**Step 4 - 发布前再次审批**：
> "修改后的文案再次经过 `interrupt_after=["generate"]` 的审批。这次审批员通过了。然后图走到 `publish` 节点——但我们在 compile 时配置了 `interrupt_before=["publish"]`，所以在正式发布前，还有一次最终确认。"

**Step 5 - Hooks 演示**：
> "再看 Pre/Post Hooks。每次调用 LLM 前，Pre-hook 自动注入『品牌规范』：『我们是高端家电品牌，语气要优雅，不要用网络梗』。LLM 输出后，Post-hook 自动检查敏感词和合规性。"

## 9. 常见误区与注意事项

| 坑 | 现象 | 预警话术 |
|----|------|---------|
| **忘记 checkpointer** | interrupt 后无法恢复，State 丢失 | "HITL 必须配 checkpointer！没有持久化，中断后就丢失了。" |
| **interrupt 和异常混用** | 用 raise Exception 代替 interrupt | "Exception 会终止执行，interrupt 是暂停。HITL 必须用 `interrupt()` 原语。" |
| **resume 值格式不对** | 被中断节点收不到 resume 值 | "resume 可以是任意类型，但要和被中断节点的期望输入匹配。" |
| **Hooks 里修改 State** | Pre-hook 改了 State，影响后续节点 | "Pre-hook 应该用 `llm_input_messages` 临时注入，不要直接改 State。" |
| **审批流设计太复杂** | 一个流程里中断 5 次，用户体验差 | "HITL 是『安全阀』不是『每一步都问』。只在高风险/高成本操作前中断。" |

### 调试技巧
1. **模拟审批**：开发时用 `Command(resume=...)` 模拟不同审批结果
2. **查看中断状态**：`graph.get_state(config)` 查看当前是否处于中断状态
3. **审计日志**：所有 HITL 操作（谁、什么时候、做了什么决定）记录到 Store

## 10. 阶段小结

### 核心要点回顾

1. **`interrupt()` = 暂停**：图在节点内暂停，等待人类输入
2. **`interrupt_before/after` = 声明式中断**：在 compile 时配置，无需改节点代码
3. **`Command(resume)` = 继续**：人类输入后恢复执行，resume 值传给被中断的节点
4. **必须配 checkpointer**：没有持久化，中断后 State 丢失，无法恢复
5. **Pre-hook = 预处理**：LLM 调用前注入上下文、安全检查、消息裁剪
6. **Post-hook = 后处理**：LLM 输出后审查、格式化、工具审批
7. **`llm_input_messages` = 临时注入**：不修改 State，仅用于本次 LLM 调用
8. **HITL 设计原则**：只在关键时刻中断，审批流要简洁

### 记忆口诀

> **"interrupt 是暂停键，resume 是播放键；Hooks 是耳机和滤镜，HITL 是安全网。"**

### 速查表

| 功能 | 方式 | 使用场景 | 代码示例 |
|------|------|---------|---------|
| 节点内中断 | `interrupt(payload)` | 条件触发审批 | `feedback = interrupt({"action": "approve"})` |
| 节点前中断 | `interrupt_before=["node"]` | 执行前审批 | `compile(..., interrupt_before=["publish"])` |
| 节点后中断 | `interrupt_after=["node"]` | 执行后确认 | `compile(..., interrupt_after=["generate"])` |
| 恢复执行 | `Command(resume=...)` | 人工输入后继续 | `graph.invoke(Command(resume={"approved": True}), config)` |
| 消息预处理 | `pre_model_hook` | 裁剪、注入上下文 | `create_react_agent(..., pre_model_hook=hook)` |
| 响应后处理 | `post_model_hook` | 审查、审批 | `create_react_agent(..., post_model_hook=hook)` |
| 临时上下文 | `llm_input_messages` | 不污染 state | `return {"llm_input_messages": [...]}` |

## 11. 课后练习

1. **编辑后重新提交**：为 ContentGuard 增加"编辑后重新提交"功能——审批员选择"修改"后，AI 根据反馈修改，然后再次进入审批流程
2. **全局审批节点**：使用 `interrupt_before` 实现一个全局审批节点，在任意敏感操作前暂停
3. **输出安全过滤**：设计一个 `post_model_hook` 实现输出安全过滤，检测并替换敏感词
4. **多轮审批**：实现一个需要"初审 → 复审"两轮的审批流程，不同角色有不同的审批权限
5. **Hooks 迁移**：将一个手动处理消息裁剪和上下文注入的节点，迁移到使用 Pre/Post Hooks 的实现

### 思考题

1. 中断后的 State 还在吗？为什么必须配 checkpointer？
2. `interrupt_before` 和 `interrupt_after` 分别适用于什么场景？
3. `messages` 和 `llm_input_messages` 的核心区别是什么？什么时候用哪个？
4. 为什么 Post-Model Hook 可以支持中断，而 Pre-Model Hook 不支持？
5. 设计一个电商场景：用户下单 → AI 生成订单 → 人工确认金额 → 支付。画出图结构并标注中断点。

---

**下一节**: LG-04 持久化与记忆系统